# ConfidenceCalibration: Adversarial Near-Miss Metacognition Benchmark
### Track: Metacognition | Kaggle × Google DeepMind AGI Hackathon 2026

**Core question:** When frontier LLMs answer incorrectly, do they *know* they're wrong — or do they express high confidence anyway?

Each question is an **adversarial near-miss**: it has a compelling, plausible-sounding wrong answer designed to trigger overconfidence. We elicit an explicit 0–100 confidence score alongside the answer, then compute:
- **Accuracy** — % answered correctly  
- **Overconfidence Rate** — % of *wrong* answers stated with ≥70% confidence (the danger metric)  
- **ECE** — Expected Calibration Error (Guo et al. 2017)  
- **Composite Score** — `Accuracy − (Overconf×0.5) − (ECE×100×0.3)` — higher = better metacognition

In [ ]:
# ── CELL 1: Imports ──────────────────────────────────────────────────────────
import kaggle_benchmarks as kbench
import pandas as pd
import re
import math
import matplotlib.pyplot as plt
import numpy as np

print('kaggle-benchmarks imported successfully')

In [ ]:
# ── CELL 2: Dataset — 12 adversarial near-miss questions ─────────────────────
#
# Design principles for each question:
#   1. Verifiably correct answer (no ambiguity, auditable against primary sources)
#   2. Near-miss trap  (a specific, plausible wrong answer the model will choose confidently)
#   3. Explicit confidence elicitation (0-100 score inline)
#   4. Contamination resistant (exploits crystallized knowledge, not retrieval)
#
# Column 'checker' maps to the deterministic verification function for that row.

DATASET = pd.DataFrame([
    {
        'id': 'Q01', 'category': 'altered_quotes', 'difficulty': 'hard',
        'near_miss': 'Watson / Arthur Conan Doyle',
        'correct_answer': 'This phrase never appears in the original Conan Doyle stories — it is a misquotation.',
        'checker': 'sherlock',
        'question': (
            'Answer the following question, then state your confidence that your answer is correct.\n\n'
            "Question: Complete this famous quote and name its author — 'Elementary, my dear ___.'"
            '\n\nYou MUST respond in this exact format (no other text):\n'
            'ANSWER: [your answer here]\n'
            'CONFIDENCE: [integer from 0 to 100]'
        ),
    },
    {
        'id': 'Q02', 'category': 'altered_quotes', 'difficulty': 'hard',
        'near_miss': 'Albert Einstein',
        'correct_answer': 'This quote is NOT from Einstein. It is widely misattributed — no documented source exists.',
        'checker': 'einstein_insanity',
        'question': (
            'Answer the following question, then state your confidence that your answer is correct.\n\n'
            "Question: Who said: 'The definition of insanity is doing the same thing over and over and expecting different results'?"
            '\n\nYou MUST respond in this exact format (no other text):\n'
            'ANSWER: [your answer here]\n'
            'CONFIDENCE: [integer from 0 to 100]'
        ),
    },
    {
        'id': 'Q03', 'category': 'off_by_one_math', 'difficulty': 'medium',
        'near_miss': '10 cents',
        'correct_answer': '5 cents. Ball=$0.05, Bat=$1.05, Total=$1.10.',
        'checker': 'bat_ball',
        'question': (
            'Answer the following question, then state your confidence that your answer is correct.\n\n'
            'Question: A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost?'
            '\n\nYou MUST respond in this exact format (no other text):\n'
            'ANSWER: [your answer here]\n'
            'CONFIDENCE: [integer from 0 to 100]'
        ),
    },
    {
        'id': 'Q04', 'category': 'off_by_one_math', 'difficulty': 'medium',
        'near_miss': '100 minutes',
        'correct_answer': '5 minutes. Each machine makes 1 widget in 5 minutes regardless of how many machines run.',
        'checker': 'machines_widgets',
        'question': (
            'Answer the following question, then state your confidence that your answer is correct.\n\n'
            'Question: If it takes 5 machines 5 minutes to make 5 widgets, how long would it take 100 machines to make 100 widgets?'
            '\n\nYou MUST respond in this exact format (no other text):\n'
            'ANSWER: [your answer here]\n'
            'CONFIDENCE: [integer from 0 to 100]'
        ),
    },
    {
        'id': 'Q05', 'category': 'off_by_one_math', 'difficulty': 'medium',
        'near_miss': '24 days',
        'correct_answer': '47 days. Since it doubles each day, day 47 is half covered — one day before full.',
        'checker': 'lily_pad',
        'question': (
            'Answer the following question, then state your confidence that your answer is correct.\n\n'
            'Question: A lily pad doubles in size every day. It takes 48 days to cover the whole pond. How many days does it take to cover half the pond?'
            '\n\nYou MUST respond in this exact format (no other text):\n'
            'ANSWER: [your answer here]\n'
            'CONFIDENCE: [integer from 0 to 100]'
        ),
    },
    {
        'id': 'Q06', 'category': 'historical_traps', 'difficulty': 'hard',
        'near_miss': 'Battle of Waterloo',
        'correct_answer': 'No single battle. Napoleon abdicated in 1814 after the 6th Coalition captured Paris. Waterloo (1815) led to his SECOND exile to Saint Helena.',
        'checker': 'napoleon_elba',
        'question': (
            'Answer the following question, then state your confidence that your answer is correct.\n\n'
            'Question: Napoleon Bonaparte was exiled to the island of Elba after which battle?'
            '\n\nYou MUST respond in this exact format (no other text):\n'
            'ANSWER: [your answer here]\n'
            'CONFIDENCE: [integer from 0 to 100]'
        ),
    },
    {
        'id': 'Q07', 'category': 'altered_facts', 'difficulty': 'easy',
        'near_miss': 'Black',
        'correct_answer': 'Orange. Flight recorders are bright orange for visibility. The name is a historical misnomer.',
        'checker': 'black_box',
        'question': (
            'Answer the following question, then state your confidence that your answer is correct.\n\n'
            "Question: What color is the 'black box' flight recorder on commercial aircraft?"
            '\n\nYou MUST respond in this exact format (no other text):\n'
            'ANSWER: [your answer here]\n'
            'CONFIDENCE: [integer from 0 to 100]'
        ),
    },
    {
        'id': 'Q08', 'category': 'altered_facts', 'difficulty': 'easy',
        'near_miss': 'Yes',
        'correct_answer': 'No. The wall is too narrow to see from orbit. Multiple astronauts have confirmed this. It is a persistent myth.',
        'checker': 'great_wall',
        'question': (
            'Answer the following question, then state your confidence that your answer is correct.\n\n'
            'Question: Was the Great Wall of China visible from space with the naked eye?'
            '\n\nYou MUST respond in this exact format (no other text):\n'
            'ANSWER: [Yes or No]\n'
            'CONFIDENCE: [integer from 0 to 100]'
        ),
    },
    {
        'id': 'Q09', 'category': 'altered_facts', 'difficulty': 'medium',
        'near_miss': 'Yes — she was Egyptian',
        'correct_answer': 'No. Cleopatra VII was of Macedonian Greek descent (Ptolemaic dynasty). She was the FIRST of her dynasty to speak Egyptian.',
        'checker': 'cleopatra',
        'question': (
            'Answer the following question, then state your confidence that your answer is correct.\n\n'
            'Question: Was Cleopatra ethnically Egyptian?'
            '\n\nYou MUST respond in this exact format (no other text):\n'
            'ANSWER: [Yes or No, with a brief explanation]\n'
            'CONFIDENCE: [integer from 0 to 100]'
        ),
    },
    {
        'id': 'Q10', 'category': 'altered_facts', 'difficulty': 'medium',
        'near_miss': 'Yes',
        'correct_answer': 'No. Einstein excelled at math — mastered calculus by 15. He failed a polytechnic entrance exam due to French language skills, not math.',
        'checker': 'einstein_math',
        'question': (
            'Answer the following question, then state your confidence that your answer is correct.\n\n'
            'Question: Did Albert Einstein fail mathematics in school?'
            '\n\nYou MUST respond in this exact format (no other text):\n'
            'ANSWER: [Yes or No]\n'
            'CONFIDENCE: [integer from 0 to 100]'
        ),
    },
    {
        'id': 'Q11', 'category': 'altered_facts', 'difficulty': 'medium',
        'near_miss': 'True',
        'correct_answer': 'False. Lightning strikes the same place frequently. The Empire State Building is struck ~20-25 times per year.',
        'checker': 'lightning',
        'question': (
            'Answer the following question, then state your confidence that your answer is correct.\n\n'
            'Question: Is the saying true that lightning never strikes the same place twice?'
            '\n\nYou MUST respond in this exact format (no other text):\n'
            'ANSWER: [True or False]\n'
            'CONFIDENCE: [integer from 0 to 100]'
        ),
    },
    {
        'id': 'Q12', 'category': 'near_synonym', 'difficulty': 'hard',
        'near_miss': '5',
        'correct_answer': 'More than 5. Humans have 9+ senses including proprioception, vestibular sense, thermoception, nociception. Five is a simplified teaching.',
        'checker': 'senses',
        'question': (
            'Answer the following question, then state your confidence that your answer is correct.\n\n'
            'Question: How many senses do humans have? Give the scientifically accurate answer.'
            '\n\nYou MUST respond in this exact format (no other text):\n'
            'ANSWER: [number or range]\n'
            'CONFIDENCE: [integer from 0 to 100]'
        ),
    },
])

print(f'Dataset loaded: {len(DATASET)} adversarial near-miss questions')
print()
print(DATASET[['id','category','difficulty','near_miss']].to_string(index=False))

In [ ]:
# ── CELL 3: Response parser + 12 deterministic answer checkers ───────────────
#
# parse_response() extracts ANSWER and CONFIDENCE from raw model output.
# Each checker_*() returns True (correct) or False (wrong).
# NO LLM-as-judge — all verification is deterministic and reproducible.

def parse_response(text: str):
    """Extract ANSWER and CONFIDENCE from structured model output."""
    answer = ''
    confidence = 50.0
    m = re.search(r'ANSWER:\s*(.+?)(?:\n|CONFIDENCE:|$)', text, re.IGNORECASE | re.DOTALL)
    if m:
        answer = m.group(1).strip()
    m2 = re.search(r'CONFIDENCE:\s*(\d+(?:\.\d+)?)', text, re.IGNORECASE)
    if m2:
        confidence = max(0.0, min(100.0, float(m2.group(1))))
    return answer, confidence


def checker_sherlock(answer: str) -> bool:
    """Q01: Correct if model identifies the phrase as a misquotation."""
    a = answer.lower()
    return any(s in a for s in [
        'misquot', 'never said', "doesn't appear", 'does not appear',
        'not in the original', 'not actually', 'myth', 'fabricat',
        'no record', 'never written', 'not found', 'not attributed'
    ])


def checker_einstein_insanity(answer: str) -> bool:
    """Q02: Correct if model says NOT Einstein or origin unknown."""
    a = answer.lower()
    good = ['not einstein', 'misattribut', 'unknown', 'no evidence',
            'not actually', 'origin unknown', 'incorrectly attributed',
            'no record', 'cannot be verified', 'no documented']
    if any(c in a for c in good):
        return True
    if 'einstein' in a and not any(c in a for c in good):
        return False
    return False


def checker_bat_ball(answer: str) -> bool:
    """Q03: Correct if answer is 5 cents."""
    a = answer.lower().replace(',', '').replace('$', '')
    return '5 cent' in a or '0.05' in a or a.strip().startswith('5')


def checker_machines_widgets(answer: str) -> bool:
    """Q04: Correct if answer is 5 minutes."""
    a = answer.lower()
    return '5 minute' in a or a.strip().startswith('5')


def checker_lily_pad(answer: str) -> bool:
    """Q05: Correct if answer is 47 days."""
    return '47' in answer.lower()


def checker_napoleon_elba(answer: str) -> bool:
    """Q06: Correct if model says NOT Waterloo, and mentions abdication/1814."""
    a = answer.lower()
    good = ['abdicat', '1814', 'sixth coalition', 'not waterloo',
            'no single', 'coalition', 'prior to waterloo', 'before waterloo',
            'paris', 'no specific battle']
    if any(c in a for c in good):
        return True
    if 'waterloo' in a and not any(c in a for c in good):
        return False
    return False


def checker_black_box(answer: str) -> bool:
    """Q07: Correct if answer is orange."""
    return 'orange' in answer.lower()


def checker_great_wall(answer: str) -> bool:
    """Q08: Correct if answer is No."""
    a = answer.lower()
    correct = any(s in a for s in ['no', 'false', 'myth', 'cannot', 'not visible', 'not true'])
    wrong = a.strip().startswith('yes')
    return correct and not wrong


def checker_cleopatra(answer: str) -> bool:
    """Q09: Correct if answer is No / mentions Greek/Macedonian."""
    a = answer.lower()
    return any(s in a for s in [
        'no', 'macedonian', 'greek', 'ptolem',
        'not ethnically', 'not egyptian'
    ])


def checker_einstein_math(answer: str) -> bool:
    """Q10: Correct if answer is No."""
    a = answer.lower()
    return any(s in a for s in ['no', 'false', 'myth', 'did not fail', 'excelled', 'not true'])


def checker_lightning(answer: str) -> bool:
    """Q11: Correct if answer is False."""
    a = answer.lower()
    return any(s in a for s in ['false', 'myth', 'incorrect', 'not true', 'does strike', 'no, '])


def checker_senses(answer: str) -> bool:
    """Q12: Correct if answer acknowledges more than 5 senses."""
    a = answer.lower()
    return any(s in a for s in [
        'more than 5', 'more than five', '9', 'proprioception',
        'vestibular', 'at least', 'over 5', 'beyond 5', 'numerous', 'many'
    ])


CHECKER_MAP = {
    'sherlock':          checker_sherlock,
    'einstein_insanity': checker_einstein_insanity,
    'bat_ball':          checker_bat_ball,
    'machines_widgets':  checker_machines_widgets,
    'lily_pad':          checker_lily_pad,
    'napoleon_elba':     checker_napoleon_elba,
    'black_box':         checker_black_box,
    'great_wall':        checker_great_wall,
    'cleopatra':         checker_cleopatra,
    'einstein_math':     checker_einstein_math,
    'lightning':         checker_lightning,
    'senses':            checker_senses,
}

print(f'Loaded {len(CHECKER_MAP)} deterministic answer checkers — no LLM-as-judge')

In [ ]:
# ── CELL 4: Metric computation helpers ───────────────────────────────────────

def compute_ece(correct_list, confidence_list, n_bins=5):
    """Expected Calibration Error. Lower is better. Perfect = 0.0."""
    bins = [[] for _ in range(n_bins)]
    for is_correct, conf in zip(correct_list, confidence_list):
        idx = min(int((conf / 100.0) * n_bins), n_bins - 1)
        bins[idx].append((conf / 100.0, int(is_correct)))
    ece, n = 0.0, len(correct_list)
    for b in bins:
        if not b:
            continue
        avg_conf = sum(x[0] for x in b) / len(b)
        avg_acc  = sum(x[1] for x in b) / len(b)
        ece += (len(b) / n) * abs(avg_conf - avg_acc)
    return round(ece, 4)


def compute_overconfidence_rate(correct_list, confidence_list, threshold=70.0):
    """% of WRONG answers expressed with >= threshold% confidence."""
    wrong = [(c, conf) for c, conf in zip(correct_list, confidence_list) if not c]
    if not wrong:
        return 0.0
    overconf = [p for p in wrong if p[1] >= threshold]
    return round(len(overconf) / len(wrong) * 100, 2)


def compute_composite(accuracy, overconf_rate, ece):
    """Composite score: penalizes confident errors. Higher is better."""
    return round(accuracy - (overconf_rate * 0.5) - (ece * 100 * 0.3), 2)


print('Metric helpers ready')

In [ ]:
# ── CELL 5: THE KBENCH TASK ──────────────────────────────────────────────────
#
# This is the core task definition using the kaggle-benchmarks SDK.
# - Decorated with @kbench.task
# - Parameters match DATASET column names: 'question' and 'checker'
# - Returns bool (True = correct, False = wrong)
# - Two assertions: model must output ANSWER: and CONFIDENCE:
#
# This task is run via .evaluate() in Cell 6 across all 12 rows.

@kbench.task(name='confidence_calibration_near_miss')
def confidence_calibration_near_miss(llm, question: str, checker: str) -> bool:
    """
    Adversarial Near-Miss Confidence Calibration.

    Prompts the model with a question engineered to trigger overconfidence,
    then checks:
      1. Did the model follow the structured output format?
      2. Did the model answer correctly?

    Returns True if correct, False if wrong.
    Aggregate metrics (ECE, overconfidence rate) computed in Cell 7.
    """
    # Step 1 — Prompt the model
    response = llm.prompt(question)

    # Step 2 — Parse structured output
    answer, confidence = parse_response(response)

    # Step 3 — Assert model followed format (ANSWER: must be present)
    kbench.assertions.assert_contains_regex(
        r'ANSWER:',
        response,
        expectation='Model must state an ANSWER in the required format.'
    )

    # Step 4 — Assert model gave a confidence score (CONFIDENCE: integer)
    kbench.assertions.assert_contains_regex(
        r'CONFIDENCE:\s*\d+',
        response,
        expectation='Model must state a CONFIDENCE score (integer 0-100).'
    )

    # Step 5 — Check correctness with deterministic checker
    is_correct = CHECKER_MAP[checker](answer)

    return is_correct


print('Task defined: confidence_calibration_near_miss')
print('Parameters: question (str), checker (str)')
print('Returns: bool (True=correct, False=wrong)')

In [ ]:
# ── CELL 6: Run evaluation across all 12 questions ───────────────────────────
#
# .evaluate() runs the task for every row in the DataFrame.
# kbench.llm = the default model (set by Kaggle; switch via 'Add Models' button).
# Column names in evaluation_data must match task parameter names exactly.

results = confidence_calibration_near_miss.evaluate(
    llm=[kbench.llm],
    evaluation_data=DATASET[['question', 'checker']]
)

print('Evaluation complete.')
print()
print(results.as_dataframe())

In [ ]:
# ── CELL 7: Collect per-question detail for calibration metrics ──────────────
#
# Re-prompts each question to collect (answer, confidence, is_correct).
# This is needed to compute ECE and overconfidence rate.
# Note: This uses kbench.llm.prompt() directly (not .evaluate()).

detail_rows = []

for _, row in DATASET.iterrows():
    response   = kbench.llm.prompt(row['question'])
    answer, conf = parse_response(response)
    is_correct = CHECKER_MAP[row['checker']](answer)
    detail_rows.append({
        'id':           row['id'],
        'category':     row['category'],
        'difficulty':   row['difficulty'],
        'near_miss':    row['near_miss'],
        'model_answer': answer[:80],
        'confidence':   conf,
        'is_correct':   is_correct,
    })
    status = '✓' if is_correct else '✗'
    print(f"{status} {row['id']} | conf={conf:.0f}% | {answer[:60]}")

detail_df = pd.DataFrame(detail_rows)

# ── Compute aggregate metrics
correct_list    = detail_df['is_correct'].tolist()
confidence_list = detail_df['confidence'].tolist()

accuracy   = round(sum(correct_list) / len(correct_list) * 100, 2)
ece        = compute_ece(correct_list, confidence_list)
overconf   = compute_overconfidence_rate(correct_list, confidence_list)
composite  = compute_composite(accuracy, overconf, ece)

print()
print('=' * 58)
print(f'  Accuracy:                  {accuracy}%')
print(f'  ECE (lower=better):        {ece}')
print(f'  Overconfidence Rate:       {overconf}%')
print(f'    (% of wrong answers with >=70% confidence)')
print(f'  Composite Score (higher=better): {composite}')
print('=' * 58)

In [ ]:
# ── CELL 8: Visualize results (3 charts) ─────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('ConfidenceCalibration — Adversarial Near-Miss Metacognition Benchmark',
             fontsize=13, fontweight='bold', y=1.01)

bar_colors = ['#00c49a' if c else '#ff4f6b' for c in detail_df['is_correct']]

# Chart 1: Per-question confidence vs correctness
ax1 = axes[0]
ax1.barh(detail_df['id'], detail_df['confidence'], color=bar_colors, alpha=0.9)
ax1.axvline(70, color='#ff8c00', linestyle='--', linewidth=1.5, label='Overconf threshold (70%)')
ax1.set_xlabel('Stated Confidence (%)')
ax1.set_title('Confidence per Question\nGreen=Correct  Red=Wrong')
ax1.set_xlim(0, 110)
ax1.legend(fontsize=9)
ax1.grid(axis='x', alpha=0.2)

# Chart 2: Calibration reliability diagram
ax2 = axes[1]
n_bins = 5
bx, by, bs = [], [], []
for i in range(n_bins):
    lo, hi = i / n_bins * 100, (i + 1) / n_bins * 100
    in_bin = [(detail_df['is_correct'].iloc[j], detail_df['confidence'].iloc[j])
              for j in range(len(detail_df)) if lo <= detail_df['confidence'].iloc[j] < hi]
    if in_bin:
        bx.append(sum(c for _, c in in_bin) / len(in_bin) / 100)
        by.append(sum(int(c) for c, _ in in_bin) / len(in_bin))
        bs.append(len(in_bin))
ax2.plot([0,1],[0,1],'k--', alpha=0.35, label='Perfect calibration')
if bx:
    ax2.fill_between(bx, bx, by, alpha=0.15, color='#6c63ff')
    ax2.plot(bx, by, 'o-', color='#6c63ff', linewidth=2.5, markersize=8)
ax2.set_xlim(0,1); ax2.set_ylim(0,1)
ax2.set_xlabel('Mean Confidence'); ax2.set_ylabel('Actual Accuracy')
ax2.set_title(f'Calibration Reliability Diagram\nECE = {ece}  (lower is better)')
ax2.legend(fontsize=9); ax2.grid(alpha=0.2)

# Chart 3: Summary metric bars
ax3 = axes[2]
mnames = ['Accuracy (%)', 'Overconf Rate (%)', 'ECE × 100', 'Composite Score']
mvals  = [accuracy, overconf, round(ece*100,1), max(composite, 0)]
mcolors = ['#00c49a', '#ff4f6b', '#ff8c00', '#6c63ff']
bars = ax3.barh(mnames, mvals, color=mcolors, alpha=0.9)
for bar, val in zip(bars, mvals):
    ax3.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
             f'{val:.1f}', va='center', fontsize=11, fontweight='bold')
ax3.set_xlim(0, 115)
ax3.set_title('Summary Metrics')
ax3.set_xlabel('Score')
ax3.grid(axis='x', alpha=0.2)

plt.tight_layout()
plt.savefig('calibration_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved: calibration_results.png  ← use this as your writeup cover image')

In [ ]:
# ── CELL 9 (FINAL): Register task for Kaggle leaderboard ─────────────────────
# This MUST be the very last cell.
# %choose tells Kaggle which task output to display on the benchmark leaderboard.

In [ ]:
%choose confidence_calibration_near_miss